# Fit data feedback

Fit modelo Nelder–Mead

In [ ]:
import numpy as np
import matplotlib as plt

In [ ]:
# --- frequency axis ---
f = f1a
w = 2*np.pi*f
# --- time delay (fixed) ---
tau = 2.38e-6
# --- convert y to linear W (you were already fitting in W) ---
P_lin = dbm_to_watts(d1a)

# --- instrument widths from RBW (Gaussian IF) ---
RBW = 100e3
VBW = 3e3   # not used by the model
sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f                # rad/s  (good initial guess)

# ---------- model pieces ----------
def S_cont(w, P0, tau_c, Omega, tau):
    x_plus  = w + Omega
    x_minus = w - Omega

    def term(x):
        den  = 1.0 + (x*tau_c)**2
        pref = 0.5 * (P0**2) * tau_c / den
        expo = np.exp(-np.abs(tau)/tau_c)
        # correct trig arguments: x in rad/s, tau in s
        bracket = 1.0 - expo*(np.cos(x*np.abs(tau)) + np.sin(x*np.abs(tau))/(x*tau_c))
        # safe x→0 limit
        lim = 1.0 - expo*(1.0 + np.abs(tau)/tau_c)
        bracket = np.where(np.isfinite(bracket), bracket, lim)
        return pref*bracket

    return term(x_minus) + term(x_plus)

def gaussian(w, mu, sigma):
    return np.exp(-0.5*((w-mu)/sigma)**2)/(np.sqrt(2*np.pi)*sigma)

# Original MODEL B, but we'll fix P0, Omega, tau and only fit [tau_c, A_g, sigma_g]
def model_B_fixed(w, tau_c, A_g, sigma_g, P0_fix, Omega_fix, tau):
    # minimal positivity clamp (since Nelder–Mead has no bounds)
    tau_c   = max(tau_c,   1e-12)
    A_g     = max(A_g,     0.0)
    sigma_g = max(sigma_g, 1e-12)
    return S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g * gaussian(w, Omega_fix, sigma_g)

# --- fix P0 and Omega from the data (use LINEAR to locate peak) ---
Omega_fix = 2*np.pi * f[np.argmax(P_lin)]
P0_fix    = np.sqrt(np.max(P_lin))  # heuristic scale (same as your code)

# ---------- objective for Nelder–Mead ----------
# We fit in linear W to match your previous code. If you prefer PSD, divide both data and model by ENBW.
def sse_theta(theta):
    tau_c, A_g, sigma_g = theta
    pred = model_B_fixed(w, tau_c, A_g, sigma_g, P0_fix, Omega_fix, tau)
    # SSE in linear W
    return np.sum((P_lin - pred)**2)

# ---------- initial guess (same spirit as your curve_fit p0) ----------
theta0 = np.array([1e-5, dbm_to_watts(np.max(d1a)), sigma_res])  # [tau_c, A_g, sigma_g]

# ---------- Nelder–Mead minimize ----------
res = minimize(sse_theta, theta0, method='Nelder-Mead',
               options=dict(maxiter=20000, xatol=1e-12, fatol=1e-12, disp=False))

tau_c_nm, A_g_nm, sigma_g_nm = res.x
fit_B_nm = model_B_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, P0_fix, Omega_fix, tau)

# ---------- plot in dBm (your original display) ----------
plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data", linewidth=2.5, color=cel)
plt.plot(f, watts_to_dbm(fit_B_nm), label="Gaussian fit (Nelder–Mead, P0/Ω/τ fixed)",
         linewidth=2.5, linestyle='--', color=ros)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (dBm)")
plt.title("Curva 1 — Nelder–Mead")
plt.grid(True)
plt.legend()
plt.show()

print("Nelder–Mead result:")
print("  tau_c      =", tau_c_nm, "s")
print("  A_g (area) =", A_g_nm,   "W")
print("  sigma_g    =", sigma_g_nm, "rad/s   (~", sigma_g_nm/(2*np.pi), "Hz)")
print("  Omega/2π   =", Omega_fix/(2*np.pi), "Hz   (fixed)")
print("  P0 (lin)   =", P0_fix, "(fixed)")
print("  success:", res.success, "  message:", res.message)


Fit dbm feedback

In [ ]:
# Fit 1

tau = 2.38e-6
df1 = f1a
f = f1a
dBm = d1a
w = 2*np.pi*f

RBW = 100e3                      # Hz
k_enbw = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW = k_enbw * RBW   

P_W = dbm_to_watts(d1a)
Sdata = P_W / ENBW 

sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f  

Omega0 = 2 * np.pi * f[np.argmax(P_W)]
P0_fix    = np.sqrt(np.max(Sdata))   
Omega_fix = 2*np.pi * f[np.argmax(P_W)]

FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G):
    S = G*S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g*gaussian(w, Omega_fix, sigma_g)
    return S + (B0 + B1*(w - w.mean()))                # W/Hz

def obj(theta):
    tau_c, A_g, sigma_g, B0, B1, G, C_dB, S_dB = theta
    # positivity clamps for NM
    tau_c   = max(tau_c,   1e-12);  A_g = max(A_g, 0.0);  sigma_g = max(sigma_g, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit   = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)      # W/Hz
        P_fit_W = S_fit*ENBW                                             # W
        m_dbm   = watts_to_dbm(P_fit_W)
        m_dbm  += C_dB + S_dB*(f - f.mean())                             # tiny dB alignment
        return np.sum((d1a - m_dbm)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)        # W/Hz
        return np.sum((Sdata - S_fit)**2)

theta0 = np.array([
    1e-5, P_W.max(), sigma_res, np.median(Sdata), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj, theta0, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit   = model_psd_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W = S_fit*ENBW
fit_dBm = watts_to_dbm(P_fit_W) + (C_dB_nm + S_dB_nm*(f - f.mean()))

plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data feedback", lw=2.5, color='teal')
plt.plot(f, fit_dBm, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 1 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:

P_W   = dbm_to_watts(d1a)     # W in RBW
Sdata = P_W           # W/Hz
Omega_fix = 2*np.pi * f[np.argmax(d1a)]
P0_fix    = np.sqrt(np.max(Sdata))

In [ ]:
FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G):
    S = G*S_cont(w, P0_fix, tau_c, Omega_fix, tau) + A_g*gaussian(w, Omega_fix, sigma_g)
    return S + (B0 + B1*(w - w.mean()))                # W/Hz

def obj(theta):
    tau_c, A_g, sigma_g, B0, B1, G, C_dB, S_dB = theta
    # positivity clamps for NM
    tau_c   = max(tau_c,   1e-12);  A_g = max(A_g, 0.0);  sigma_g = max(sigma_g, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit   = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)      # W/Hz
        P_fit_W = S_fit*ENBW                                             # W
        m_dbm   = watts_to_dbm(P_fit_W)
        m_dbm  += C_dB + S_dB*(f - f.mean())                             # tiny dB alignment
        return np.sum((d1a - m_dbm)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit = model_psd_fixed(w, tau_c, A_g, sigma_g, B0, B1, G)        # W/Hz
        return np.sum((Sdata - S_fit)**2)

theta0 = np.array([
    1e-5, P_W.max(), sigma_res, np.median(Sdata), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj, theta0, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit   = model_psd_fixed(w, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W = S_fit*ENBW
fit_dBm = watts_to_dbm(P_fit_W) + (C_dB_nm + S_dB_nm*(f - f.mean()))

In [ ]:
# --- choose a trace ---
tau = 2.38e-6

df1 = f1a
f1 = df1

w1 = 2*np.pi*f1                    # rad/s                # rad/s

dBm1 = d1a                        # analyzer trace (power in RBW)

RBW = 100e3                      # Hz
k_enbw = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW = k_enbw * RBW   

# Convert y (dBm) to linear power (W). Fit in linear space for stability.
PWa = dbm_to_watts(d1a)

Sdata1 = PWa / ENBW 

sigma_f   = RBW / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res = 2*np.pi * sigma_f  

# ---- initial guesses (tweak if needed) ----
# Omega guess from where the peak is (in rad/s):
Omega01 = 2 * np.pi * f[np.argmax(PWa)]

P0_fixa    = np.sqrt(np.max(Sdata1))        # scale heuristic for S_cont      # scale heuristic for S_cont

thetaa = np.array([ 1e-6, PWa.max(), sigma_res, np.median(Sdata1), 0.0, 1.0, 0.0, 0.0 ])

res1 = minimize(obj, thetaa, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma, C_dB_nma, S_dB_nma = res1.x


# plot exactly like the analyzer (no surprises)
S_fit1   = model_psd_fixed(w1, tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma)

P_fit_W1 = S_fit1*ENBW


fit_dBm1 = watts_to_dbm(P_fit_W1) + (C_dB_nma + S_dB_nma*(f1 - f1.mean()))
plt.figure(figsize=(9,5))
plt.plot(f, d1a, label="data", lw=2.5, color='teal')
plt.plot(f, fit_dBm, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 1 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:
#plot -------------------------------------------------------------------------
fig, ax = plt.subplots(1,1, figsize =(6,4))

ax.plot(f1a*1e-6, d1a, c=cel, linewidth=2.5, alpha=0.8, label = 'Feedback')
ax.plot(f1a*1e-6, fit_dBm1, label="fit", linewidth=2.5, linestyle = '--', alpha=0.6, color = ros)

ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Signal (dBm)')

#tau c
ax.annotate(f"$\\tau_c = {tau_c_nm:.3g}$" + r"s", xy=(0.65,0.9), xycoords='axes fraction', textcoords='offset points',size=14,
            bbox=dict(boxstyle="round", fc=ros, alpha=0.5, ec="none"))

# Legend
ax.legend()

#ax.set_title('Fitting with feedback and coherence time')
plt.savefig('fitting_feedback_coherence_time.png', format='png', dpi=1200)
fig.tight_layout()
plt.show()

# Fit data No feedback

Fit modelo Nelder–Mead

In [ ]:
# --- frequency axis ---
f2 = f2a
w2 = 2*np.pi*f2                   # rad/s
# --- time delay (fixed) ---
tau2 = 2.38e-6
# --- convert y to linear W (you were already fitting in W) ---
P_lin2 = dbm_to_watts(d2a)

# --- instrument widths from RBW (Gaussian IF) ---
RBW2 = 100e3
VBW2 = 3e3   # not used by the model
sigma_f2   = RBW2 / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res2 = 2*np.pi * sigma_f2                # rad/s  (good initial guess)

# ---------- model pieces ----------
def S_cont2(w2, P02, tau_c2, Omega2, tau2):
    x_plus  = w2 + Omega2
    x_minus = w2 - Omega2

    def term(x):
        den  = 1.0 + (x*tau_c2)**2
        pref = 0.5 * (P02**2) * tau_c2 / den
        expo = np.exp(-np.abs(tau2)/tau_c2)
        # correct trig arguments: x in rad/s, tau in s
        bracket = 1.0 - expo*(np.cos(x*np.abs(tau2)) + np.sin(x*np.abs(tau2))/(x*tau_c2))
        # safe x→0 limit
        lim = 1.0 - expo*(1.0 + np.abs(tau2)/tau_c2)
        bracket = np.where(np.isfinite(bracket), bracket, lim)
        return pref*bracket

    return term(x_minus) + term(x_plus)

def gaussian(w2, mu2, sigma2):
    return np.exp(-0.5*((w2-mu2)/sigma2)**2)/(np.sqrt(2*np.pi)*sigma2)

# Original MODEL B, but we'll fix P0, Omega, tau and only fit [tau_c, A_g, sigma_g]
def model_B_fixed2(w2, tau_c2, A_g2, sigma_g2, P0_fix, Omega_fix, tau2):
    # minimal positivity clamp (since Nelder–Mead has no bounds)
    tau_c2  = max(tau_c2,   1e-12)
    A_g2    = max(A_g2,     0.0)
    sigma_g2 = max(sigma_g2, 1e-12)
    return S_cont2(w2, P0_fix, tau_c2, Omega_fix, tau2) + A_g2 * gaussian(w2, Omega_fix, sigma_g2)

# --- fix P0 and Omega from the data (use LINEAR to locate peak) ---
Omega_fix2 = 2*np.pi * f[np.argmax(P_lin2)]
P0_fix2    = np.sqrt(np.max(P_lin2))  # heuristic scale (same as your code)

# ---------- objective for Nelder–Mead ----------
# We fit in linear W to match your previous code. If you prefer PSD, divide both data and model by ENBW.
def sse_theta(theta2):
    tau_c2, A_g2, sigma_g2 = theta2
    pred2 = model_B_fixed2(w2, tau_c2, A_g2, sigma_g2, P0_fix2, Omega_fix2, tau2)
    # SSE in linear W
    return np.sum((P_lin2- pred2)**2)

# ---------- initial guess (same spirit as your curve_fit p0) ----------
theta02 = np.array([1e-5, dbm_to_watts(np.max(d2a)), sigma_res2])  # [tau_c, A_g, sigma_g]

# ---------- Nelder–Mead minimize ----------
res = minimize(sse_theta, theta02, method='Nelder-Mead',
               options=dict(maxiter=20000, xatol=1e-12, fatol=1e-12, disp=False))

tau_c_nm, A_g_nm, sigma_g_nm = res.x
fit_B_nm = model_B_fixed2(w2, tau_c_nm, A_g_nm, sigma_g_nm, P0_fix2, Omega_fix2, tau2)

# ---------- plot in dBm (your original display) ----------
plt.figure(figsize=(9,5))
plt.plot(f2, d2a, label="data no feedback", linewidth=2.5, color=cel)
plt.plot(f2, watts_to_dbm(fit_B_nm), label="Gaussian fit (Nelder–Mead, P0/Ω/τ fixed)",
         linewidth=2.5, linestyle='--', color=ros)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (dBm)")
plt.title("Curva 1 — Nelder–Mead")
plt.grid(True)
plt.legend()
plt.show()

print("Nelder–Mead result:")
print("  tau_c      =", tau_c_nm, "s")
print("  A_g (area) =", A_g_nm,   "W")
print("  sigma_g    =", sigma_g_nm, "rad/s   (~", sigma_g_nm/(2*np.pi), "Hz)")
print("  Omega/2π   =", Omega_fix/(2*np.pi), "Hz   (fixed)")
print("  P0 (lin)   =", P0_fix, "(fixed)")
print("  success:", res.success, "  message:", res.message)


Fit dbm feedback

In [ ]:
# Fit 1

tau2 = 2.38e-6
df2 = f2a
f2 = f2a
dBm2 = d2a
w2 = 2*np.pi*f2

RBW2 = 100e3                      # Hz
k_enbw2 = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW2 = k_enbw2 * RBW2

P_W2 = dbm_to_watts(dBm2)
Sdata2 = P_W2 / ENBW2

sigma_f2   = RBW2 / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res2 = 2*np.pi * sigma_f2

Omega02 = 2 * np.pi * f2[np.argmax(P_W2)]
P0_fix2    = np.sqrt(np.max(Sdata2))   
Omega_fix2 = 2*np.pi * f2[np.argmax(P_W2)]

FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2):
    S = G2*S_cont(w2, P0_fix2, tau_c2, Omega_fix2, tau2) + A_g2*gaussian(w2, Omega_fix2, sigma_g2)
    return S + (B02 + B12*(w2 - w2.mean()))                # W/Hz

def obj(theta2):
    tau_c2, A_g2, sigma_g2, B02, B12, G2, C_dB2, S_dB2 = theta2
    # positivity clamps for NM
    tau_c2   = max(tau_c2,   1e-12);  A_g2 = max(A_g2, 0.0);  sigma_g2 = max(sigma_g2, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit2   = model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2)      # W/Hz
        P_fit_W2 = S_fit2*ENBW                                             # W
        m_dbm2   = watts_to_dbm(P_fit_W2)
        m_dbm2  += C_dB2 + S_dB2*(f2 - f2.mean())                             # tiny dB alignment
        return np.sum((d2a - m_dbm2)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit2 = model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2)        # W/Hz
        return np.sum((Sdata2 - S_fit2)**2)

theta02 = np.array([
    1e-5, P_W2.max(), sigma_res2, np.median(Sdata2), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj, theta02, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit2   = model_psd_fixed2(w2, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W2 = S_fit2*ENBW2
fit_dBm2 = watts_to_dbm(P_fit_W2) + (C_dB_nm + S_dB_nm*(f2 - f2.mean()))

plt.figure(figsize=(9,5))
plt.plot(f2, d2a, label="data no feedback", lw=2.5, color='teal')
plt.plot(f2, fit_dBm2, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 1 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:

P_W2   = dbm_to_watts(d2a)     # W in RBW
Sdata2 = P_W2           # W/Hz}
Omega_fix2 = 2*np.pi * f2[np.argmax(d2a)]
P0_fix2    = np.sqrt(np.max(Sdata2))

In [ ]:
FIT_IN_DBM = True  # <- flip to False for linear-PSD fitting

def model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2):
    S2 = G2*S_cont(w2, P0_fix2, tau_c2, Omega_fix2, tau2) + A_g2*gaussian(w2, Omega_fix2, sigma_g2)
    return S2 + (B02 + B12*(w2 - w2.mean()))                # W/Hz

def obj2(theta2):
    tau_c2, A_g2, sigma_g2, B02, B12, G2, C_dB2, S_dB2 = theta2
    # positivity clamps for NM
    tau_c2   = max(tau_c2,   1e-12);  A_g2 = max(A_g2, 0.0);  sigma_g2 = max(sigma_g2, 1e-12)

    if FIT_IN_DBM:
        # model -> W in RBW -> dBm, then allow small offset/slope in dB (display alignment)
        S_fit2   = model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2)      # W/Hz
        P_fit_W2 = S_fit2*ENBW2                                             # W
        m_dbm2   = watts_to_dbm(P_fit_W2)
        m_dbm2  += C_dB2 + S_dB2*(f2 - f2.mean())                             # tiny dB alignment
        return np.sum((d2a - m_dbm2)**2)
    else:
        # linear PSD fitting (physically clean)
        S_fit2 = model_psd_fixed2(w2, tau_c2, A_g2, sigma_g2, B02, B12, G2)        # W/Hz
        return np.sum((Sdata2 - S_fit2)**2)

theta02 = np.array([
    1e-5, P_W2.max(), sigma_res2, np.median(Sdata2), 0.0, 1.0, 0.0, 0.0   # + C_dB, S_dB
])

res = minimize(obj2, theta02, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm, C_dB_nm, S_dB_nm = res.x

# plot exactly like the analyzer (no surprises)
S_fit2   = model_psd_fixed2(w2, tau_c_nm, A_g_nm, sigma_g_nm, B0_nm, B1_nm, G_nm)
P_fit_W2 = S_fit2*ENBW2
fit_dBm2 = watts_to_dbm(P_fit_W2) + (C_dB_nm + S_dB_nm*(f2 - f2.mean()))

In [ ]:
# --- choose a trace ---
tau2 = 2.38e-6

df2 = f2a
f2 = df2

w2 = 2*np.pi*f2                    # rad/s                # rad/s

dBm2 = d2a                        # analyzer trace (power in RBW)

RBW2 = 100e3                      # Hz
k_enbw2 = 1.06                    # ~Gaussian IF; adjust if your manual says otherwise
ENBW2 = k_enbw2 * RBW2

# Convert y (dBm) to linear power (W). Fit in linear space for stability.
PWa2 = dbm_to_watts(d2a)

Sdata2 = PWa2 / ENBW 

sigma_f2   = RBW2 / (2*np.sqrt(2*np.log(2)))   # Hz
sigma_res2 = 2*np.pi * sigma_f2

# ---- initial guesses (tweak if needed) ----
# Omega guess from where the peak is (in rad/s):
Omega02 = 2 * np.pi * f2[np.argmax(PWa2)]

P0_fixa2    = np.sqrt(np.max(Sdata2))        # scale heuristic for S_cont      # scale heuristic for S_cont

thetaa2 = np.array([ 1e-6, PWa2.max(), sigma_res2, np.median(Sdata2), 0.0, 1.0, 0.0, 0.0 ])

res2 = minimize(obj2, thetaa2, method="Nelder-Mead",
               options=dict(maxiter=30000, xatol=1e-12, fatol=1e-12))
tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma, C_dB_nma, S_dB_nma = res2.x


# plot exactly like the analyzer (no surprises)
S_fit2   = model_psd_fixed(w2, tau_c_nma, A_g_nma, sigma_g_nma, B0_nma, B1_nma, G_nma)

P_fit_W2 = S_fit2*ENBW2


fit_dBm2 = watts_to_dbm(P_fit_W2) + (C_dB_nma + S_dB_nma*(f2 - f2.mean()))
plt.figure(figsize=(9,5))
plt.plot(f2, d2a, label="data", lw=2.5, color='teal')
plt.plot(f2, fit_dBm2, label="NM fit (in dBm)", lw=2.5, ls='--', color='red')
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (dBm)"); plt.title("Curva 2 — Nelder–Mead (dBm fit)")
plt.grid(True); plt.legend(); plt.show()

In [ ]:
#plot -------------------------------------------------------------------------
fig, ax = plt.subplots(1,1, figsize =(6,4))

ax.plot(f2a*1e-6, d2a, lw=3, c=cel, alpha=0.5, label = 'No feedback')
ax.plot(f2a*1e-6, fit_dBm2, label="fit", linewidth=2.5, linestyle = '--', alpha=0.6, color = ros)

ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Signal (dBm)')

#tau c
ax.annotate(f"$\\tau_c = {tau_c_nma:.3g}$"+ r"s", xy=(0.68,0.9), xycoords='axes fraction', textcoords='offset points',size=14,
            bbox=dict(boxstyle="round", fc=ros, alpha=0.5, ec="none"))

# Legend
ax.legend()
#ax.set_title('Fitting with no feedback and coherence time')
plt.savefig('fitting_no_feedback_coherence_time.png', format='png', dpi=1200)
fig.tight_layout()
plt.show()

In [ ]:
print(tau_c_nm)

In [ ]:
print(tau_c_nma)

# ancho de linea

In [ ]:
print('feedback=',(1./(tau_c_nm)    ) ,' Hz')
print('no feedback=',(1./(tau_c_nma)),'Hz')

# Simulacioncita

In [ ]:
def power_spectral_density(f,tau,tau_c, tauexp, P0=1, Omega=54e6, delta_width=1.0):
    """
    Power spectral density model based on the paper's equation.

    Parameters:
    f: frequency array (Hz)
    tau: time delay (s)
    tau_c: coherence time (s)
    P0: laser power (W)
    Omega: AOM frequency (Hz)
    delta_width: width for delta approximation (same units as omega - Omega, i.e. rad/s)
    """
    # do not overwrite the passed tau; convert frequencies to angular frequency
    omega = 2 * np.pi * f
    Omega_rad = 2 * np.pi * Omega
    resta = omega - Omega_rad
    suma = omega + Omega_rad
    
    tauexp = (np.abs(tau)/tau_c)

    # Main Lorentzian terms
    lorentzian1 = (0.5 * P0**2 * tau_c) / (1.0 + (resta)**2 * tau_c**2)
    lorentzian2 = (0.5 * P0**2 * tau_c) / (1.0 + (suma)**2 * tau_c**2)

    # Oscillation prefactor
    exp_factor = np.exp(-tauexp)  # exp(-|τ|/τ_c)

    # safe bracket calculation with proper x->0 limit
    def bracket_for(x):
        cos_term = np.cos(x * np.abs(tau))
        with np.errstate(divide='ignore', invalid='ignore'):
            sin_term = np.sin(x * np.abs(tau)) / (x * tau_c)
        # safe x->0 limit: 1 - exp_factor*(1 + |tau|/tau_c)
        lim = 1.0 - exp_factor * (1.0 + np.abs(tau) / tau_c)
        bracket = 1.0 - exp_factor * (cos_term + sin_term)
        bracket = np.where(np.isfinite(bracket), bracket, lim)
        return bracket

    bracket1 = bracket_for(resta)
    bracket2 = bracket_for(suma)

    # Narrow Lorentzian approximation for delta functions (regularized)
    delta_amplitude = 0.5 * P0**2 * np.pi * exp_factor
    # use delta_width as small width in rad/s (avoid calling it like a function)
    delta_term1 = delta_amplitude * (delta_width / (np.pi * (delta_width**2 + resta**2)))
    delta_term2 = delta_amplitude * (delta_width / (np.pi * (delta_width**2 + suma**2)))

    # Complete PSD
    S = lorentzian1 * bracket1 + lorentzian2 * bracket2 + delta_term1 + delta_term2

    return S

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
tau_values = [1.5e-6, 3.5e-6, 10e-6, 30e-6]
tau_c = 4e-6  # coherence time
P0 = 1
Omega = 54e6  # AOM frequency   
cmap = plt.get_cmap('cool')
colors = [cmap(i /len(tau_values)) for i in range(len(tau_values))]  # Generate different shades

f = np.linspace(Omega - 0.5e6, Omega + 0.5e6, 2000)

for i, (tau, color) in enumerate(zip(tau_values, colors)):
    S = power_spectral_density(f, tau, tau_c, tauexp=np.abs(tau)/tau_c, P0=P0, Omega=Omega, delta_width=1000)
    S_db = 10 * np.log10(S / np.max(S))
    
    ax.plot((f - Omega)/1e3, S_db, c=color, linewidth=2.5)
    ax.set_xlabel('Frequency (kHz)')
    ax.set_ylabel('Normalized PSD (dBm)')
    ax.set_xlim(-530, 530)
    
ax.legend([f'$|\\tau|/\\tau_c$ = {tau*1e6:.1f}' for tau in tau_values], frameon=False)
 
plt.savefig('psd_vs_delay.png', format='png', dpi=1200)   
plt.tight_layout()
plt.show()